# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashishpal003/flyrank_ml_intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Connect + assemble the eligible, labeled, feature slice at D

One DuckDB query (an extension of the ML-04 contract's assembly cell) builds one row per eligible page at `D = 2026-03-01`, carrying: 90-day and windowed GSC aggregates (impressions, clicks, impression-weighted position, impression-days), a little `dim_content` context for display, and the forward label — computed with the **exact same expression as ML-04**.

In [1]:
%pip -q install duckdb huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, getpass
from pathlib import Path

# Token order: env var -> .env file (local runs) -> Colab Secret -> prompt (last resort).
# Never commit the token: `.env` is gitignored and this repo is public.
def _from_dotenv(key):
    for base in [Path.cwd(), *Path.cwd().parents]:
        f = base / ".env"
        if f.is_file():
            for line in f.read_text().splitlines():
                s = line.strip()
                if s.startswith(f"{key}=") or s.startswith(f"export {key}="):
                    return s.split("=", 1)[1].strip().strip('\"').strip("'")
    return None

HF_TOKEN = os.environ.get("HF_TOKEN") or _from_dotenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "no valid HF READ token found (env / .env / Colab secret)"

In [3]:
import duckdb, json
import numpy as np
import pandas as pd

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

D = '2026-03-01'
FEATURE_START = '2025-12-01'         # D - 90d
LABEL_END = '2026-03-31'            # D + 30d
FEATURE_MONTHS = ['2025-12', '2026-01', '2026-02']
LABEL_MONTHS = ['2026-03']
K_ABS = 50                          # primary editor budget
SEED = 42

def daily(months):
    """read_parquet over an explicit list of month partitions (hf:// has no brace globs)."""
    paths = [f"'{REL}/fact_content_daily_performance/month={m}/*.parquet'" for m in months]
    return f"read_parquet([{', '.join(paths)}])"

print('connected. D =', D, '| feature [', FEATURE_START, ',', D, ') | label [', D, ',', LABEL_END, ']')

connected. D = 2026-03-01 | feature [ 2025-12-01 , 2026-03-01 ) | label [ 2026-03-01 , 2026-03-31 ]


In [4]:
raw = con.sql(f"""
    WITH feat AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions)                                                                   AS imp_90d,
               SUM(f.gsc_clicks)                                                                        AS clk_90d,
               SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions  ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_impressions  ELSE 0 END) AS imp_prev60,
               SUM(CASE WHEN f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_sum_position ELSE 0 END) AS sumpos_last30,
               SUM(CASE WHEN f.report_date <  DATE '{D}' - INTERVAL 30 DAY THEN f.gsc_sum_position ELSE 0 END) AS sumpos_prev60,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0
                     AND f.report_date >= DATE '{D}' - INTERVAL 30 DAY THEN f.report_date END)           AS days_impr_last30,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0
                     AND f.report_date <  DATE '{D}' - INTERVAL 30 DAY
                     AND f.report_date >= DATE '{D}' - INTERVAL 60 DAY THEN f.report_date END)           AS days_impr_prev30
        FROM {daily(FEATURE_MONTHS)} f
        WHERE f.report_date >= DATE '{FEATURE_START}' AND f.report_date < DATE '{D}'
        GROUP BY 1, 2
    ),
    lab AS (
        SELECT content_hash_id,
               SUM(gsc_impressions)                                                                     AS imp_label,
               SUM(CASE WHEN report_date <  DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h1,
               SUM(CASE WHEN report_date >= DATE '{D}' + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_label_h2
        FROM {daily(LABEL_MONTHS)}
        WHERE report_date >= DATE '{D}' AND report_date <= DATE '{LABEL_END}'
        GROUP BY 1
    )
    SELECT feat.*,
           dc.word_count, dc.content_type, dc.main_intent,
           dc.content_updated_date,
           CASE WHEN dc.content_updated_date < DATE '{D}'
                THEN date_diff('day', dc.content_updated_date, DATE '{D}') END                           AS days_since_update,
           cl.gsc_data_start,
           feat.imp_90d / 3.0                                                                           AS pace_30d,  -- feature-only: the 90d run-rate over 30 days
           CASE WHEN feat.imp_last30 > 0 THEN feat.sumpos_last30::DOUBLE / feat.imp_last30 END           AS pos_last30,
           CASE WHEN feat.imp_prev60 > 0 THEN feat.sumpos_prev60::DOUBLE / feat.imp_prev60 END           AS pos_prev60,
           COALESCE(lab.imp_label, 0)    AS imp_label,
           COALESCE(lab.imp_label_h1, 0) AS imp_label_h1,
           COALESCE(lab.imp_label_h2, 0) AS imp_label_h2
    FROM feat
    LEFT JOIN {DIM_CONTENT} dc ON feat.content_hash_id = dc.content_hash_id
    LEFT JOIN {DIM_CLIENTS} cl ON feat.client_hash_id = cl.client_hash_id
    LEFT JOIN lab              ON feat.content_hash_id = lab.content_hash_id
""").df()

# --- eligibility filter + forward label: identical logic to ML-04 cell-6d ---
raw['pass_volume_floor']   = raw['imp_last30'] >= 100
raw['pass_client_history'] = raw['gsc_data_start'] <= (pd.Timestamp(D) - pd.Timedelta(days=90))
raw['pass_not_freefall']   = ~((raw['imp_prev60'] > 0) & (raw['imp_last30'] < 0.5 * raw['imp_prev60']))
raw['label_decline'] = ((raw['imp_label'] < 0.75 * raw['pace_30d']) &
                        (raw['imp_label_h2'] <= raw['imp_label_h1'])).astype(int)

funnel = {
    'content_in_feature_window': len(raw),
    'after_volume_floor': int(raw['pass_volume_floor'].sum()),
    'after_client_history': int((raw['pass_volume_floor'] & raw['pass_client_history']).sum()),
    'after_exclude_freefall': int((raw['pass_volume_floor'] & raw['pass_client_history'] & raw['pass_not_freefall']).sum()),
}
elig = raw[raw['pass_volume_floor'] & raw['pass_client_history'] & raw['pass_not_freefall']].reset_index(drop=True).copy()
BASE_RATE = float(elig['label_decline'].mean())

for k, v in funnel.items():
    print(f'{k:28} {v:>8,}')
print(f'\neligible rows: {len(elig):,}   forward-decline base rate: {BASE_RATE:.4f}')
assert abs(len(elig) - 60_265) <= 50 and abs(BASE_RATE - 0.072) < 0.01, 'does not reconcile with ml04_contract_checks.json'
print('reconciles with ML-04 (60,265 eligible, base rate ~0.072)')


content_in_feature_window     321,546
after_volume_floor             81,521
after_client_history           73,326
after_exclude_freefall         60,265

eligible rows: 60,265   forward-decline base rate: 0.0717
reconciles with ML-04 (60,265 eligible, base rate ~0.072)


## 1. My rule and its reason codes

**The rule, in three sentences.** A page belongs on this week's review queue if the search traffic it earned over the last 90 days is now running below its own recent pace, its average ranking position has slipped over the last 30 days, and the number of days it shows up in search at all is thinning. All three signals are measured **strictly before `D`** — nothing from the label window, no FlyRank product flag, no query-mix table. Exposure (`imp_90d`) is shown in the queue so an editor can see what is at stake, and fires a `high_exposure` reason code, but it is **not scored** — the signal check below shows bigger pages actually decline *less*, so scoring it would point the wrong way.

**Transparent score** — three components, each a 0–1 percentile rank, combined with **hand-set weights that are never fitted** (shape follows `scripts/02_baseline_score.py`; `pct_rank` = `Series.rank(pct=True)`):

| component | meaning | definition | weight |
|---|---|---|---|
| `traffic_softening` | recent 30d below the 90d pace | `pct_rank(clip(1 - imp_last30 / (imp_90d/3), 0, 1))` | 0.40 |
| `position_slip` | avg position got worse, last 30d vs prior 60d | `pct_rank(clip(pos_last30 - pos_prev60, 0, 20))`, only where both defined | 0.30 |
| `reach_thinning` | fewer days with any impressions | `pct_rank(clip(1 - days_impr_last30 / days_impr_prev30, 0, 1))` | 0.30 |

`baseline_score = 0.40·traffic_softening + 0.30·position_slip + 0.30·reach_thinning`

**Reason codes** (plain-unit thresholds; every scored page carries the list, `|`-joined; fallback `general_review`):

| code | fires when |
|---|---|
| `high_exposure` | `imp_90d >= 500` *(annotation only — not scored)* |
| `traffic_softening` | `imp_last30 < 0.8 * imp_90d/3` |
| `position_slipping` | `pos_last30 - pos_prev60 >= 1.0` (both defined) |
| `reach_thinning` | `days_impr_last30 <= 0.75 * days_impr_prev30` |
| `stale_content` | `content_updated_date < D` **and** `days_since_update >= 180` — *annotation only*: ML-04 showed edit-date coverage before `D` is partial |

`suggested_action`: has `position_slipping` or `traffic_softening` → `refresh`; only `stale_content` → `refresh_metadata`; else → `monitor`.

The cell below scores **all four candidate signals** against the label by quartile (a compact signal check), which is what justified dropping `visibility` from the score before freezing the rule."

In [5]:
def pct_rank(s):
    return s.rank(pct=True, method='average')

def components_from(df):
    """Candidate signals, computed only from columns knowable strictly before D."""
    p = (df['imp_90d'] / 3.0).replace(0, np.nan)
    pg = df['pos_last30'] - df['pos_prev60']
    return pd.DataFrame({
        'visibility':        pct_rank(np.log1p(df['imp_90d'])),
        'traffic_softening': pct_rank((1 - df['imp_last30'] / p).clip(0, 1).fillna(0)),
        'position_slip':     pct_rank(pg.clip(0, 20).fillna(0)) * pg.notna().astype(int),
        'reach_thinning':    pct_rank((1 - df['days_impr_last30'] / df['days_impr_prev30'].replace(0, np.nan)).clip(0, 1).fillna(0)),
    }, index=df.index)

e = elig
comp = components_from(e)
for c in comp.columns:
    e[c] = comp[c]

CANDIDATES = list(comp.columns)
WEIGHTS = {'traffic_softening': 0.40, 'position_slip': 0.30, 'reach_thinning': 0.30}   # visibility dropped -- see below
COMPONENTS = list(WEIGHTS)

# --- compact signal check: forward-decline rate by candidate-signal quartile ---
rows = []
for c in CANDIDATES:
    q = pd.qcut(e[c].rank(method='first'), 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
    rate = e.groupby(q, observed=True)['label_decline'].mean()
    rows.append({'signal': c, 'Q1': rate['Q1'], 'Q2': rate['Q2'], 'Q3': rate['Q3'], 'Q4': rate['Q4'],
                 'Q4_minus_Q1': rate['Q4'] - rate['Q1'],
                 'scored': c in WEIGHTS,
                 'points_wrong_way': (rate['Q4'] - rate['Q1']) < 0})
signal_check = pd.DataFrame(rows).set_index('signal').round(4)
print(f'forward-decline rate by signal quartile (base rate = {BASE_RATE:.4f}):\n')
print(signal_check.to_string())
wrong = list(signal_check.index[signal_check['points_wrong_way']])
if wrong:
    print(f'\n-> {wrong} points the WRONG way (higher value = LESS decline); excluded from the score,'
          '\n   kept only as a `high_exposure` reason-code annotation.')


forward-decline rate by signal quartile (base rate = 0.0717):

                       Q1      Q2      Q3      Q4  Q4_minus_Q1  scored  points_wrong_way
signal                                                                                  
visibility         0.1124  0.0834  0.0501  0.0407      -0.0717   False              True
traffic_softening  0.0709  0.0739  0.0686  0.0731       0.0021    True             False
position_slip      0.0565  0.0776  0.0686  0.0839       0.0274    True             False
reach_thinning     0.0481  0.0492  0.0483  0.1410       0.0930    True             False

-> ['visibility'] points the WRONG way (higher value = LESS decline); excluded from the score,
   kept only as a `high_exposure` reason-code annotation.


## 2. Build the ranked queue (writes the CSV)

Score every eligible page, attach reason codes and a suggested action, rank, and write `work/outputs/baseline_action_score.csv` (a working artifact — gitignored). Then evaluate on the **same slice + forward label** against two references:

- **FlyRank's stale-visible rule** — `(days_since_update >= 180) AND (imp_90d >= 500)`, ranked by `imp_90d` (the rule from `docs/ml-intern-dataset-and-lane-guide.md` / `scripts/02_baseline_score.py`);
- **random** — a seeded shuffle, the floor.

Metric: **precision@K** at `K = 50` (primary editor budget) and `K = 5%` of the eligible set, with **recall@K** and **average precision**, all printed next to the base rate. `work/outputs/ml07_baseline_metrics.json` is the committed receipt.

In [6]:
# --- score (ties broken by exposure: a documented ops choice, not a fitted term) ---
e['baseline_score'] = (sum(WEIGHTS[c] * e[c] for c in COMPONENTS)
                       + 1e-9 * pct_rank(np.log1p(e['imp_90d'])))

def reason_codes(r):
    out = []
    if r['imp_90d'] >= 500: out.append('high_exposure')
    if r['imp_last30'] < 0.8 * (r['imp_90d'] / 3.0): out.append('traffic_softening')
    if pd.notna(r['pos_last30']) and pd.notna(r['pos_prev60']) and (r['pos_last30'] - r['pos_prev60']) >= 1.0:
        out.append('position_slipping')
    if r['days_impr_prev30'] > 0 and r['days_impr_last30'] <= 0.75 * r['days_impr_prev30']:
        out.append('reach_thinning')
    if pd.notna(r['days_since_update']) and r['days_since_update'] >= 180:
        out.append('stale_content')
    return out or ['general_review']

def suggested_action(codes):
    if 'position_slipping' in codes or 'traffic_softening' in codes: return 'refresh'
    if codes == ['stale_content']: return 'refresh_metadata'
    return 'monitor'

_codes = e.apply(reason_codes, axis=1)
e['reason_codes'] = _codes.apply('|'.join)
e['suggested_action'] = _codes.apply(suggested_action)
e['baseline_rank'] = e['baseline_score'].rank(method='first', ascending=False).astype(int)

# --- comparison rankings on the SAME slice ---
e['stale_visible_score'] = (((e['days_since_update'].fillna(-1) >= 180) & (e['imp_90d'] >= 500)).astype(int) * e['imp_90d'])
rng = np.random.default_rng(SEED)
e['random_score'] = rng.random(len(e))

# --- metrics ---
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    return float(np.asarray(labels)[order[:k]].mean())

def recall_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    lab = np.asarray(labels)
    return float(lab[order[:k]].sum() / max(lab.sum(), 1))

def average_precision(scores, labels):
    order = np.argsort(-np.asarray(scores, dtype=float), kind='stable')
    yy = np.asarray(labels)[order]
    if yy.sum() == 0: return 0.0
    prec = np.cumsum(yy) / (np.arange(len(yy)) + 1)
    return float((prec * yy).sum() / yy.sum())

y = e['label_decline'].to_numpy()
K_PCT = max(1, round(0.05 * len(e)))
rankers = {'baseline_rule': e['baseline_score'], 'stale_visible_rule': e['stale_visible_score'], 'random': e['random_score']}

metric_rows = []
for name, sc in rankers.items():
    metric_rows.append({
        'ranker': name,
        f'precision@{K_ABS}': precision_at_k(sc, y, K_ABS),
        f'precision@{K_PCT}(5%)': precision_at_k(sc, y, K_PCT),
        f'recall@{K_ABS}': recall_at_k(sc, y, K_ABS),
        'avg_precision': average_precision(sc, y),
    })
metrics = pd.DataFrame(metric_rows).set_index('ranker').round(4)
print(f'base rate (random pick) = {BASE_RATE:.4f}   |   K = {K_ABS} and {K_PCT} (5% of {len(e):,})\n')
print(metrics.to_string())

# --- honesty checks on the headline number ---
order50 = np.argsort(-e['baseline_score'].to_numpy(), kind='stable')[:K_ABS]
top50_clients = e.iloc[order50]['client_hash_id'].nunique()
top50_top_client_share = e.iloc[order50]['client_hash_id'].value_counts(normalize=True).iloc[0]
def per_client_p_at(df, score_col, k=10):
    vals = [precision_at_k(g[score_col], g['label_decline'], k)
            for _, g in df.groupby('client_hash_id') if len(g) >= k]
    return float(np.mean(vals)) if vals else float('nan')
pc_baseline = per_client_p_at(e, 'baseline_score')
pc_stale = per_client_p_at(e, 'stale_visible_score')
print(f'\nCONCENTRATION: the top {K_ABS} come from only {top50_clients} client(s); '
      f'largest single client = {top50_top_client_share:.0%} of them')
print(f'per-client precision@10 (mean over clients w/ >=10 eligible): baseline {pc_baseline:.3f} | '
      f'stale-visible {pc_stale:.3f} | base rate {BASE_RATE:.3f}')


base rate (random pick) = 0.0717   |   K = 50 and 3013 (5% of 60,265)

                    precision@50  precision@3013(5%)  recall@50  avg_precision
ranker                                                                        
baseline_rule               0.98              0.3170     0.0113         0.2727
stale_visible_rule          0.84              0.0840     0.0097         0.0831
random                      0.10              0.0694     0.0012         0.0720

CONCENTRATION: the top 50 come from only 2 client(s); largest single client = 98% of them
per-client precision@10 (mean over clients w/ >=10 eligible): baseline 0.295 | stale-visible 0.238 | base rate 0.072


In [7]:
# --- write the ranked queue (working artifact; work/**/*.csv is gitignored) ---
OUT = None
for cand in [Path('work/outputs'), Path('../outputs'), Path('outputs')]:
    if cand.parent.exists():
        OUT = cand; break
OUT = OUT or Path('work/outputs')
OUT.mkdir(parents=True, exist_ok=True)

queue_cols = ['content_hash_id', 'client_hash_id', 'baseline_rank', 'baseline_score',
              *COMPONENTS, 'reason_codes', 'suggested_action', 'label_decline',
              'imp_90d', 'imp_last30', 'pace_30d', 'pos_last30', 'pos_prev60',
              'days_impr_last30', 'days_impr_prev30', 'days_since_update', 'word_count', 'content_type']
queue = e[queue_cols].sort_values('baseline_rank').reset_index(drop=True)
queue.to_csv(OUT / 'baseline_action_score.csv', index=False)
print('wrote', (OUT / 'baseline_action_score.csv').resolve(), f'({len(queue):,} rows)')

# --- write the committed receipt ---
receipt = {
    'decision_date': D,
    'eligible_rows': int(len(e)),
    'base_rate': round(BASE_RATE, 4),
    'score_formula': WEIGHTS,
    'visibility_excluded_because': 'Q4-Q1 decline lift is negative (bigger pages decline less)',
    'K': {'absolute': K_ABS, 'pct5': K_PCT},
    'metrics': {name: {k: round(v, 4) for k, v in row.items()} for name, row in metrics.T.to_dict().items()},
    'top50_distinct_clients': int(top50_clients),
    'top50_largest_client_share': round(float(top50_top_client_share), 3),
    'per_client_precision_at_10': {'baseline': round(pc_baseline, 4), 'stale_visible': round(pc_stale, 4)},
    'signal_check': signal_check[['Q4_minus_Q1', 'scored', 'points_wrong_way']].to_dict(orient='index'),
    'top20_reason_code_counts': queue.head(20)['reason_codes'].str.split('|').explode().value_counts().to_dict(),
    'top20_hit_rate': round(float(queue.head(20)['label_decline'].mean()), 4),
}
(OUT / 'ml07_baseline_metrics.json').write_text(json.dumps(receipt, indent=2, default=str))
print('wrote', (OUT / 'ml07_baseline_metrics.json').resolve())
print(json.dumps(receipt, indent=2, default=str))


wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/outputs/baseline_action_score.csv (60,265 rows)
wrote /Users/ashishpal/Documents/flyrank/flyrank_ml_intern/work/outputs/ml07_baseline_metrics.json
{
  "decision_date": "2026-03-01",
  "eligible_rows": 60265,
  "base_rate": 0.0717,
  "score_formula": {
    "traffic_softening": 0.4,
    "position_slip": 0.3,
    "reach_thinning": 0.3
  },
  "visibility_excluded_because": "Q4-Q1 decline lift is negative (bigger pages decline less)",
  "K": {
    "absolute": 50,
    "pct5": 3013
  },
  "metrics": {
    "baseline_rule": {
      "precision@50": 0.98,
      "precision@3013(5%)": 0.317,
      "recall@50": 0.0113,
      "avg_precision": 0.2727
    },
    "stale_visible_rule": {
      "precision@50": 0.84,
      "precision@3013(5%)": 0.084,
      "recall@50": 0.0097,
      "avg_precision": 0.0831
    },
    "random": {
      "precision@50": 0.1,
      "precision@3013(5%)": 0.0694,
      "recall@50": 0.0012,
      "avg_precision": 0.

## 3. Top-20 review

**Top-20 hit rate: 0.95** (19 of 20 declined) against a base rate of 0.072 — a 13× lift. Every one is actioned `refresh`. Reason codes: `position_slipping` on all 20, `high_exposure` on 16, `reach_thinning` on 5.

**But read the `client_hash_id` column: 19 of the 20 are the same client** (`client_861cdcccf8049915`). That client lost search rankings across most of its portfolio in March 2026, and the rule is really detecting *one client's bad month*, not 20 independent at-risk pages. See §4 and the concentration line in §2 — the per-client precision@10 (0.30) is the number to trust, not precision@50 (0.98).

| what the rule saw | reading |
|---|---|
| `pos_move` of +20 to +41 positions | these pages fell from roughly page 1–2 to page 3–5+ over the last 30 days |
| `last30_vs_pace` ≈ 1.0–2.3 (at or above pace) | their February **impressions** were still fine — the position drop had not yet reached traffic. This is exactly the early-warning the project is for: flagged *before* the loss shows in reporting. |
| ranks 2–5: `imp_90d` 170–280 | low-volume pages where a 20-position "move" in a mean is statistically noisy — lower confidence (and the one miss, rank 3, is this shape — see §4) |

**Confidence note.** High where `high_exposure` + `position_slipping` + `reach_thinning` all fire on a page with thousands of impressions (rank 1); low for the sub-300-impression pages near the top. **What would make the top-20 wrong:** (a) the one-client concentration — a real *per-client* weekly queue would never look like this; (b) position noise on low-volume pages; (c) if `client_861`'s March drop is a tracking artifact rather than a real decline, the whole top-20 is spurious — ML-09 must audit that client's label.

In [8]:
top20 = queue.head(20).copy()
top20['last30_vs_pace'] = (top20['imp_last30'] / top20['pace_30d']).round(2)
top20['pos_move'] = (top20['pos_last30'] - top20['pos_prev60']).round(1)
top20['n_signals'] = top20['reason_codes'].str.split('|').apply(lambda c: len([x for x in c if x != 'general_review']))
show = ['baseline_rank', 'content_hash_id', 'client_hash_id', 'baseline_score', 'reason_codes',
        'suggested_action', 'imp_90d', 'last30_vs_pace', 'pos_move', 'days_since_update',
        'n_signals', 'label_decline']
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)
print(f'top-20 hit rate = {top20["label_decline"].mean():.3f}   (base rate {BASE_RATE:.3f})\n')
print(top20[show].to_string(index=False))


top-20 hit rate = 0.950   (base rate 0.072)

 baseline_rank          content_hash_id          client_hash_id  baseline_score                                   reason_codes suggested_action  imp_90d  last30_vs_pace  pos_move  days_since_update  n_signals  label_decline
             1 content_4280ff1b4fc7724c client_861cdcccf8049915        0.797519 high_exposure|position_slipping|reach_thinning          refresh   2801.0            1.05      26.1                  4          3              1
             2 content_8e25cfe7f4899c49 client_861cdcccf8049915        0.797519               position_slipping|reach_thinning          refresh    244.0            1.33      20.8                  4          2              1
             3 content_46fd94fd6f7ad247 client_ff644d8251367cbb        0.797460               position_slipping|reach_thinning          refresh    171.0            1.75      41.5                  4          2              0
             4 content_4d0ebab0f38fee58 client_861cdcccf804

## 4. Weak picks + leakage check

**Weak pick (the one top-50 miss).** Rank 3, `content_46fd94fd6f7ad247` (`client_ff644d8251367cbb`): `imp_90d` = 171, `pos_move` = +41.5, but `softening` = −0.75 — its February impressions ran **75% above** the 90-day pace. A tiny page whose *average position* swung wildly (a handful of query impressions can move a mean by 40 places) while its actual traffic grew. The `position_slip` component, capped at 20, saturated on that noise. It did not decline. **This is the rule's characteristic failure: it has no volume guard on the position component, so low-volume pages with noisy position means float to the top.** The eligibility floor (`imp_last30 >= 100`, locked in ML-04) is too low to prevent this.

**Second weakness — ties at the top.** Ranks 6–20 all carry the identical `baseline_score` (0.795404): their `position_slip` and `reach_thinning` components are both clipped to the maximum, so `pct_rank` ties them, and only the tiny exposure tiebreak orders them. "Rank 7 vs rank 15" inside that block is not meaningful — an editor should treat the whole tied block as one bucket.

**Signal check (from §1).** `visibility` was dropped before freezing — bigger pages decline *less* here (Q4 rate 0.041 < base rate 0.072). Of the three scored signals, `reach_thinning` separates the label best (Q4 rate 0.14, ~2× base); `traffic_softening` separates it weakly (Q4−Q1 ≈ +0.001) because eligibility already removed the sharpest cases and most pages sit at the component's floor — it is kept as part of the plain-English story, and ML-08/ML-09 will judge it.

**Leakage verdict — both asserts pass.**
- **Assert 1:** the score is bit-identical when every `[D, D+30d]` column is dropped from the frame → nothing from the label window feeds it.
- **Assert 2:** no scored component correlates with the label above |r| = 0.5. `reach_thinning` is the highest at **r = +0.42** — legitimate (February impression-days vs March impressions are different windows) but the closest to concerning; ML-09 should stress-test whether "fewer impression-days just before D" is standing in for the outcome.
- Not touched anywhere: `trend_direction` / `trend_pct` / `is_declining_label`; FlyRank product flags (`health_score` / `priority_score` / `action_type`); any `fact_content_query_90d` column; `last_optimized_date`.

**This baseline is now FROZEN** — score formula, weights, the eligible slice, K = 50 / 5%, and the exact numbers in `work/outputs/ml07_baseline_metrics.json`. ML-08 compares against these; it does not get to move them.

In [9]:
# --- weak picks: pages the rule ranked at the very top that did NOT decline ---
pool = queue.head(50).copy()
pool['n_signals'] = pool['reason_codes'].str.split('|').apply(lambda c: len([x for x in c if x != 'general_review']))
pool['softening'] = (1 - pool['imp_last30'] / pool['pace_30d']).round(2)
pool['pos_move'] = (pool['pos_last30'] - pool['pos_prev60']).round(1)
misses = pool[pool['label_decline'] == 0].sort_values(['n_signals', 'softening'])
n_miss = len(misses)
print(f'top-50 misses (ranked high, did NOT decline): {n_miss}  -> precision@50 = {1 - n_miss/50:.2f}')
print(misses[['baseline_rank', 'content_hash_id', 'client_hash_id', 'reason_codes', 'imp_90d',
              'softening', 'pos_move', 'days_since_update']].to_string(index=False))
assert n_miss >= 1, 'top 50 was 100% precision -- widen the pool / look harder'

# --- leakage asserts ---
SCORE_INPUT_COLS = ['imp_90d', 'imp_last30', 'pace_30d', 'pos_last30', 'pos_prev60',
                    'days_impr_last30', 'days_impr_prev30']
BANNED = {'trend_direction', 'trend_pct', 'is_declining_label', 'health_score', 'priority_score',
          'action_type', 'refresh_tier', 'last_optimized_date',
          'imp_label', 'imp_label_h1', 'imp_label_h2'}   # imp_label* = label-window ingredients
assert not (set(queue.columns) & BANNED), f'banned column in the queue: {set(queue.columns) & BANNED}'
assert not (set(SCORE_INPUT_COLS) & BANNED)

# re-derive the score from a frame with EVERY label-window column dropped -> must match (within the tiebreak epsilon)
safe = e.drop(columns=[c for c in e.columns if c.startswith('imp_label')])
safe_comp = components_from(safe)
safe_score = sum(WEIGHTS[c] * safe_comp[c] for c in WEIGHTS) + 1e-9 * pct_rank(np.log1p(safe['imp_90d']))
assert np.allclose(safe_score.to_numpy(), e['baseline_score'].to_numpy()), 'score depends on a label-window column!'
print('\nleakage assert 1: score is identical when all [D, D+30d] columns are dropped  -> OK')

corr = {c: float(np.corrcoef(e[c], y)[0, 1]) for c in COMPONENTS}
print('leakage assert 2: |corr(scored component, label)| all < 0.5 (no component is a relabelled outcome):')
for c, r in corr.items():
    print(f'    {c:18} r = {r:+.3f}')
assert all(abs(r) < 0.5 for r in corr.values())
print('\nall leakage asserts passed -- baseline FROZEN for ML-08.')


top-50 misses (ranked high, did NOT decline): 1  -> precision@50 = 0.98
 baseline_rank          content_hash_id          client_hash_id                     reason_codes  imp_90d  softening  pos_move  days_since_update
             3 content_46fd94fd6f7ad247 client_ff644d8251367cbb position_slipping|reach_thinning    171.0      -0.75      41.5                  4

leakage assert 1: score is identical when all [D, D+30d] columns are dropped  -> OK
leakage assert 2: |corr(scored component, label)| all < 0.5 (no component is a relabelled outcome):
    traffic_softening  r = -0.000
    position_slip      r = +0.042
    reach_thinning     r = +0.416

all leakage asserts passed -- baseline FROZEN for ML-08.


## Self-check

- [x] Rule stated in three plain sentences; **three scored components, hand-set weights, nothing fitted** (`visibility` dropped by the signal check before freezing); baseline frozen for ML-08
- [x] precision@K computed on the ML-04 eligible slice + forward label (rows = 60,265, base rate = 0.072) — reconciled with `ml04_contract_checks.json`
- [x] Evaluated against the base rate **and** FlyRank's stale-visible rule **and** a random floor, at K = 50 and K = 5%; per-client precision@10 reported as the honest headline
- [x] Every scored page carries reason codes and a suggested action; ranked queue written to `work/outputs/baseline_action_score.csv`
- [x] Top-20 hand review done (§3); the single top-50 weak pick dissected in §4; client-concentration caveat stated
- [x] Leakage asserts pass: score is identical with all `[D, D+30d]` columns dropped; no product flag / `fact_content_query_90d` / `trend_*` / `last_optimized_date`; no scored component |r| > 0.5 with the label
- [x] IDs shown are hashes; no client names / URLs / raw queries; June 2026 never queried
- [ ] Commit `work/notebooks/w04_baseline_score.ipynb` (with outputs) + `work/outputs/ml07_baseline_metrics.json` (the CSV stays gitignored)